<a href="https://colab.research.google.com/github/vkhanht1/NVIDIA-NeMo-ASR-FastConformer-ONNX/blob/main/NVIDIA-NeMo-ASR-FastConformer-ONNX.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
!pip install wget
!apt-get install sox libsndfile1 ffmpeg
!pip install text-unidecode
!pip install matplotlib>=3.3.2
!pip uninstall -y nemo_toolkit pytorch-lightning lightning
!pip install nemo_toolkit[all]

BRANCH = 'main'


In [ ]:
import os
data_dir = '.'

if not os.path.exists(data_dir):
  os.makedirs(data_dir)

In [ ]:
import glob
import os
import subprocess
import tarfile
import wget

print("******")
if not os.path.exists(data_dir + '/an4_sphere.tar.gz'):
    an4_url = 'https://dldata-public.s3.us-east-2.amazonaws.com/an4_sphere.tar.gz'
    an4_path = wget.download(an4_url, data_dir)
    print(f"Dataset downloaded at: {an4_path}")
else:
    print("Tarfile already exists.")
    an4_path = data_dir + '/an4_sphere.tar.gz'

if not os.path.exists(data_dir + '/an4/'):
    tar = tarfile.open(an4_path)
    tar.extractall(path=data_dir)

    print("Converting .sph to .wav...")
    sph_list = glob.glob(data_dir + '/an4/**/*.sph', recursive=True)
    for sph_path in sph_list:
        wav_path = sph_path[:-4] + '.wav'
        cmd = ["sox", sph_path, wav_path]
        subprocess.run(cmd)
print("Finished conversion.\n******")

In [ ]:
import librosa
import IPython.display as ipd

example_file = data_dir + '/an4/wav/an4_clstk/mgah/cen2-mgah-b.wav'
audio, sample_rate = librosa.load(example_file)

ipd.Audio(example_file, rate=sample_rate)

In [ ]:
%matplotlib inline
import librosa.display
import matplotlib.pyplot as plt

plt.rcParams['figure.figsize'] = (15,7)
plt.title('Waveform of Audio Example')
plt.ylabel('Amplitude')

_ = librosa.display.waveshow(audio, color='blue')

In [ ]:
import numpy as np

spec = np.abs(librosa.stft(audio))
spec_db = librosa.amplitude_to_db(spec, ref=np.max)

librosa.display.specshow(spec_db, y_axis='log', x_axis='time')
plt.colorbar()
plt.title('Audio Spectrogram');

In [ ]:
mel_spec = librosa.feature.melspectrogram(y=audio, sr=sample_rate)
mel_spec_db = librosa.power_to_db(mel_spec, ref=np.max)

librosa.display.specshow(
    mel_spec_db, x_axis='time', y_axis='mel')
plt.colorbar()
plt.title('Mel Spectrogram');

In [ ]:
import nemo
import nemo.collections.asr as nemo_asr

In [ ]:
fastconformer = nemo_asr.models.EncDecCTCModelBPE.from_pretrained(model_name="stt_en_fastconformer_ctc_large")

In [ ]:
files = [os.path.join(data_dir, 'an4/wav/an4_clstk/mgah/cen2-mgah-b.wav')]
for fname, transcription in zip(files, fastconformer.transcribe(audio=files)):
  print(f"Audio in {fname} was recognized as: {transcription}")

In [ ]:
import json

def build_manifest(transcripts_path, manifest_path, wav_path):
    with open(transcripts_path, 'r') as fin:
        with open(manifest_path, 'w') as fout:
            for line in fin:
                transcript = line[: line.find('(')-1].lower()
                transcript = transcript.replace('<s>', '').replace('</s>', '')
                transcript = transcript.strip()

                file_id = line[line.find('(')+1 : -2]
                audio_path = os.path.join(
                    data_dir, wav_path,
                    file_id[file_id.find('-')+1 : file_id.rfind('-')],
                    file_id + '.wav')

                duration = librosa.core.get_duration(filename=audio_path)

                metadata = {
                    "audio_filepath": audio_path,
                    "duration": duration,
                    "text": transcript
                }
                json.dump(metadata, fout)
                fout.write('\n')

print("******")
train_transcripts = data_dir + '/an4/etc/an4_train.transcription'
train_manifest = data_dir + '/an4/train_manifest.json'
if not os.path.isfile(train_manifest):
    build_manifest(train_transcripts, train_manifest, 'an4/wav/an4_clstk')
    print("Training manifest created.")

test_transcripts = data_dir + '/an4/etc/an4_test.transcription'
test_manifest = data_dir + '/an4/test_manifest.json'
if not os.path.isfile(test_manifest):
    build_manifest(test_transcripts, test_manifest, 'an4/wav/an4test_clstk')
    print("Test manifest created.")
print("***Done***")

In [ ]:
try:
    from ruamel.yaml import YAML
except ModuleNotFoundError:
    from ruamel_yaml import YAML
config_path = './configs/conformer_ctc_char.yaml'

if not os.path.exists(config_path):
    BRANCH = 'main'
    !mkdir -p configs
    !wget -P configs/ https://raw.githubusercontent.com/NVIDIA/NeMo/$BRANCH/examples/asr/conf/conformer/conformer_ctc_char.yaml

yaml = YAML(typ='safe')
with open(config_path) as f:
    params = yaml.load(f)
print(params)

In [ ]:
import lightning.pytorch as pl
trainer = pl.Trainer(devices=1, accelerator='gpu', max_epochs=50)

In [ ]:
type(trainer)

In [ ]:
params['model']['labels'] = params['model']['labels']

print(params['model'].keys())

In [ ]:
from omegaconf import OmegaConf

params['model']['train_ds']['manifest_filepath'] = train_manifest
params['model']['validation_ds']['manifest_filepath'] = test_manifest

conf = OmegaConf.create(params)

first_asr_model = nemo_asr.models.EncDecCTCModel(cfg=conf.model, trainer=trainer)

In [ ]:
import nemo.collections.asr as nemo_asr

save_path = "/content/drive/MyDrive/asr_model_trained.nemo"
model_ready = nemo_asr.models.EncDecCTCModel.restore_from(save_path)

In [ ]:
trainer.fit(first_asr_model)

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
import os

save_path = "/content/drive/MyDrive/asr_model_trained.nemo"

first_asr_model.save_to(save_path)

print(f"Success save : {save_path}")

In [ ]:
!ls -lh /content/drive/MyDrive/asr_model_trained.nemo

In [ ]:
try:
  from google import colab
  COLAB_ENV = True
except (ImportError, ModuleNotFoundError):
  COLAB_ENV = False

if COLAB_ENV:
  %load_ext tensorboard
  %tensorboard --logdir lightning_logs/
else:
  print("To use tensorboard, please use this notebook in a Google Colab environment.")

In [ ]:
print(params['model']['optim'])

In [ ]:
import copy
from omegaconf import DictConfig

new_opt = copy.deepcopy(params['model']['optim'])

new_opt['sched']['d_model'] = params['model']['encoder']['d_model']

new_opt['lr'] = 0.001

first_asr_model.setup_optimization(optim_config=DictConfig(new_opt))

In [ ]:
first_asr_model.cuda()
first_asr_model.eval()
audio = [os.path.join(data_dir, 'an4/wav/an4_clstk/mgah/cen2-mgah-b.wav'),
                     os.path.join(data_dir, 'an4/wav/an4_clstk/fmjd/cen7-fmjd-b.wav'),
                     os.path.join(data_dir, 'an4/wav/an4_clstk/fmjd/cen8-fmjd-b.wav'),
                     os.path.join(data_dir, 'an4/wav/an4_clstk/fkai/cen8-fkai-b.wav')]
print(first_asr_model.transcribe(audio=audio,
                                 batch_size=4))

In [ ]:
from omegaconf import OmegaConf

conf = OmegaConf.create(params)
resolved_conf = OmegaConf.to_container(conf, resolve=True)
resolved_conf = OmegaConf.create(resolved_conf)

resolved_conf.model.validation_ds.batch_size = 16

first_asr_model.setup_test_data(test_data_config=resolved_conf.model.validation_ds)
first_asr_model.cuda()
first_asr_model.eval()

wer_nums = []
wer_denoms = []

for test_batch in first_asr_model.test_dataloader():
    test_batch = [x.cuda() for x in test_batch]
    targets = test_batch[2]
    targets_lengths = test_batch[3]
    log_probs, encoded_len, greedy_predictions = first_asr_model(
        input_signal=test_batch[0], input_signal_length=test_batch[1]
    )
    first_asr_model.wer.update(predictions=greedy_predictions, predictions_lengths=None, targets=targets, targets_lengths=targets_lengths)
    _, wer_num, wer_denom = first_asr_model.wer.compute()
    first_asr_model.wer.reset()
    wer_nums.append(wer_num.detach().cpu().numpy())
    wer_denoms.append(wer_denom.detach().cpu().numpy())

    del test_batch, log_probs, targets, targets_lengths, encoded_len, greedy_predictions

print(f"WER = {sum(wer_nums)/sum(wer_denoms)}")

In [ ]:
print(fastconformer._cfg['spec_augment'])

In [ ]:
print(fastconformer.decoder.vocabulary)

print(f"Tokenizer type: {type(fastconformer.tokenizer)}")